#### 데이터 전처리

In [47]:
# 경로
data_path = "./data/kowiki.txt"
preprocessed_data_path = "./data/preprocessed_kowiki.txt"

In [ ]:
import re


def preprocess_text(text):
    # ^기호를 통한 화이트 리스트 방식의 특수문자 필터링
    preprocessed_text = re.sub(r"[^a-zA-Z0-9가-힣?.!,()\"\'\s]", " ", text)
    preprocessed_text = re.sub(r"\s+", " ", preprocessed_text)
    return preprocessed_text

# 매니저 열고
with open(data_path, "r", encoding="utf-8") as input, \
    open(preprocessed_data_path, "w", encoding="utf-8") as output:
    # 한줄씩 전처리 적용 후 파일에 작성, 공백 행 처리 로직 x
    for line in input:
        line = preprocess_text(line)
        output.write(line + "\n")

In [ ]:
from kiwipiepy import Kiwi


# 키위 + 센텐스피스 = 국밥
kiwi = Kiwi()
def get_sentences_from_splited_chunks(input_path, output_path):
    count = 0
    article_count = 0
    
    # 텍스트 파일의 전체 내용을 메모리에 올리지 않고 chunk로 처리하는 제너레이터
    def chunk_generator(file_path):
        # 입력과 출력 사이의 속도차 및 프로세스적 완충지대, 임시적인 공간
        buffer = []
        for line in file_path:
            # 개행문자 라인인 경우
            if line == "\n":
                # buffer에 담는다.
                buffer.append(line)
                # 만약 빈 줄이 3개 이상 모였다면,
                if len(buffer) >= 3:
                    # 지금까지 쌓인 문장을 하나의 chunk로 리턴
                    pass

    # --- 실전용: 단순하지만 강력한 행 단위 처리 ---
    with open(input_path, "r", encoding="utf-8") as input, \
         open(output_path, "w", encoding="utf-8") as output:
        
        # 현재 담긴 라인들
        current_lines = []        
        for line in input:
            if line.strip(): # 내용이 있는 줄
                current_lines.append(line.strip())
            else: # 빈 줄을 만났을 때
                if current_lines:
                    # 쌓인 줄들을 하나의 덩어리로 보고 리스트의 요소들을 " "(공백)로 길게 연결한다.
                    chunk = " ".join(current_lines)
                    # 길게 연결된 청크를 키위에 넣어 split_into_sents 메서드로 문장을 구분한다.
                    sentences = kiwi.split_into_sents(chunk)
                    
                    valid_sents = []
                    for s in sentences:
                        txt = s.text.strip()
                        if 20 <= len(txt) <= 128:
                            valid_sents.append(txt)
                            count += 1
                    
                    if valid_sents:
                        output.write("\n".join(valid_sents) + "\n\n\n\n") # 문서 간 경계 복원
                        output.flush() # 즉시 파일에 기록 (기다리는 동안 확인 가능)
                    
                    current_lines = [] # 초기화
                    article_count += 1
                    if article_count % 1000 == 0:
                        print(f"{article_count}개 문서 처리 중... 추출 문장: {count}")

    print(f"최종 완료! 총 문장: {count}")

# 주의: input_path가 진짜 '원본 데이터'인지 다시 확인하세요!
get_sentences_from_splited_chunks(preprocessed_data_path, "./data/kowikiSentences_with_gaps.txt")

1000개 문서 처리 중... 추출 문장: 31993
2000개 문서 처리 중... 추출 문장: 54853
3000개 문서 처리 중... 추출 문장: 75233
4000개 문서 처리 중... 추출 문장: 106450
5000개 문서 처리 중... 추출 문장: 139944
6000개 문서 처리 중... 추출 문장: 172386
7000개 문서 처리 중... 추출 문장: 197023
8000개 문서 처리 중... 추출 문장: 200015
9000개 문서 처리 중... 추출 문장: 207171
10000개 문서 처리 중... 추출 문장: 225183
11000개 문서 처리 중... 추출 문장: 247677
12000개 문서 처리 중... 추출 문장: 277934
13000개 문서 처리 중... 추출 문장: 307485
14000개 문서 처리 중... 추출 문장: 329919
15000개 문서 처리 중... 추출 문장: 352397
16000개 문서 처리 중... 추출 문장: 373656
17000개 문서 처리 중... 추출 문장: 392501
18000개 문서 처리 중... 추출 문장: 411093
19000개 문서 처리 중... 추출 문장: 427665
20000개 문서 처리 중... 추출 문장: 438420
21000개 문서 처리 중... 추출 문장: 449920
22000개 문서 처리 중... 추출 문장: 466063
23000개 문서 처리 중... 추출 문장: 487609
24000개 문서 처리 중... 추출 문장: 504347
25000개 문서 처리 중... 추출 문장: 526241
26000개 문서 처리 중... 추출 문장: 544850
27000개 문서 처리 중... 추출 문장: 564710
28000개 문서 처리 중... 추출 문장: 584214
29000개 문서 처리 중... 추출 문장: 603528
30000개 문서 처리 중... 추출 문장: 623595
31000개 문서 처리 중... 추출 문장: 639876
32000개 문서 처리 중... 추출

In [ ]:
import numpy as np
import sentencepiece as spm

MAX_LEN = 256


def build_perfect_binary_and_map(input_path, bin_path, eod_path, tokenizer):
    cls_id = tokenizer.piece_to_id("[CLS]")
    sep_id = tokenizer.piece_to_id("[SEP]")
    
    eod_indices = []
    written_count = 0
    blank_line_count = 0
    current_article_has_sent = False 
    
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(bin_path, "wb") as f_out:
        
        for line in f_in:
            clean_line = line.strip()
            
            if not clean_line: # 빈 줄(개행) 감지
                blank_line_count += 1
                continue
            
            # 문장을 만났는데 이전에 3칸 이상의 개행(문서 경계)이 있었다면
            if blank_line_count >= 2 and current_article_has_sent:
                eod_indices.append(written_count - 1)
                current_article_has_sent = False
            
            blank_line_count = 0 # 카운터 초기화
            
            # 1. 정수 인코딩 및 물리적 길이 체크 (62토큰 이하만)
            token_ids = tokenizer.encode_as_ids(clean_line)
            
            if 16 <= len(token_ids) <= 124: # 완벽한 문장만 선별
                # 2. 스페셜 토큰 부착 및 패딩
                final_seq = [cls_id] + token_ids + [sep_id]
                padded = final_seq + [0] * (128 - len(final_seq))
                
                # 3. 바이너리 기록 (int32, 4bytes)
                f_out.write(np.array(padded, dtype=np.int32).tobytes())
                
                written_count += 1
                current_article_has_sent = True
                
                if written_count % 100000 == 0:
                    print(f"[{written_count}] 정예 문장 인덱싱 중...")

        # 파일 끝 처리
        if current_article_has_sent:
            eod_indices.append(written_count - 1)

    # 최종 지도 저장
    np.save(eod_path, np.array(eod_indices))
    print(f"최종 리포트: {written_count}개 문장 확보, {len(eod_indices)}개 문서 경계 확정.")

# 실행 예시
build_perfect_binary_and_map("./data/kowikiSentences_with_gaps.txt", "./data/kowiki_128slot_final.bin", "./data/eod_map.npy", tokenizer_model)

[100000] 정예 문장 인덱싱 중...
[200000] 정예 문장 인덱싱 중...
[300000] 정예 문장 인덱싱 중...
[400000] 정예 문장 인덱싱 중...
[500000] 정예 문장 인덱싱 중...
[600000] 정예 문장 인덱싱 중...
[700000] 정예 문장 인덱싱 중...
[800000] 정예 문장 인덱싱 중...
[900000] 정예 문장 인덱싱 중...
[1000000] 정예 문장 인덱싱 중...
[1100000] 정예 문장 인덱싱 중...
[1200000] 정예 문장 인덱싱 중...
[1300000] 정예 문장 인덱싱 중...
[1400000] 정예 문장 인덱싱 중...
[1500000] 정예 문장 인덱싱 중...
[1600000] 정예 문장 인덱싱 중...
[1700000] 정예 문장 인덱싱 중...
[1800000] 정예 문장 인덱싱 중...
[1900000] 정예 문장 인덱싱 중...
[2000000] 정예 문장 인덱싱 중...
[2100000] 정예 문장 인덱싱 중...
[2200000] 정예 문장 인덱싱 중...
[2300000] 정예 문장 인덱싱 중...
[2400000] 정예 문장 인덱싱 중...
[2500000] 정예 문장 인덱싱 중...
[2600000] 정예 문장 인덱싱 중...
[2700000] 정예 문장 인덱싱 중...
[2800000] 정예 문장 인덱싱 중...
[2900000] 정예 문장 인덱싱 중...
[3000000] 정예 문장 인덱싱 중...
[3100000] 정예 문장 인덱싱 중...
최종 리포트: 3121944개 문장 확보, 433992개 문서 경계 확정.


In [15]:
import numpy as np

def check_max_token_length(bin_path):
    # 1. 바이너리 파일을 메모리 맵으로 연결 (메모리 절약)
    data = np.fromfile(bin_path, dtype=np.int32).reshape(-1, 128)
    
    # 2. 각 행에서 0(PAD)이 아닌 토큰의 개수를 계산
    # (True는 1, False는 0으로 계산되므로 sum을 하면 길이가 나옵니다)
    lengths = (data != 0).sum(axis=1)
    
    # 3. 통계치 산출
    max_len = np.max(lengths)
    min_len = np.min(lengths)
    avg_len = np.mean(lengths)
    
    print(f"--- 데이터 무결성 리포트 ---")
    print(f"최대 토큰 길이: {max_len}")
    print(f"최소 토큰 길이: {min_len}")
    print(f"평균 토큰 길이: {avg_len:.2f}")
    
    # 4. 설계 한계점(64)을 넘는 것이 있는지 재확인
    overflow_count = np.sum(lengths > 128)
    if overflow_count == 0:
        print("결과: 모든 문장이 128슬롯 내에 완벽하게 안착했습니다. (Safe)")
    else:
        print(f"경고: {overflow_count}개의 문장이 슬롯을 초과했습니다!")

# 실행
check_max_token_length("./data/kowiki_128slot_final.bin")

--- 데이터 무결성 리포트 ---
최대 토큰 길이: 104
최소 토큰 길이: 18
평균 토큰 길이: 30.29
결과: 모든 문장이 128슬롯 내에 완벽하게 안착했습니다. (Safe)


In [11]:
import sentencepiece as spm


VOCAB_SIZE = 32000

def build_vocabulary():
    spm.SentencePieceTrainer.train(
        input="./data/kowikiSentences_with_gaps.txt",
        model_prefix="models/bertbot_spm",
        vocab_size=VOCAB_SIZE,
        pad_id=0,
        unk_id=1,
        bos_id=-1, 
        eos_id=-1,
        user_defined_symbols=["[CLS]", "[SEP]", "[MASK]"],
        model_type="bpe"
    )

    sp = spm.SentencePieceProcessor()
    sp.load("models/bertbot_spm.model")
    return sp

tokenizer_model = build_vocabulary()


In [1]:
# 리스타트 분기

In [1]:
import sentencepiece as spm


sp = spm.SentencePieceProcessor()
sp.load("./models/bertbot_spm.model")
tokenizer_model = sp

In [2]:
bin_path = "./data/kowiki_128slot_final.bin"
eod_path = "./data/eod_map.npy"

In [3]:
import torch
from torch.utils.data import Dataset
import numpy as np
import random

class BERTDataset(Dataset):
    def __init__(self, bin_path, eod_path, tokenizer, max_len=256):
        # 1. 바이너리 연결
        self.data = np.memmap(bin_path, dtype=np.int32, mode='r').reshape(-1, 128)
        self.total_sentences = self.data.shape[0]
        
        # 2. EOD 지도 로드
        self.eod_map = set(np.load(eod_path))
        
        # 3. 속성 정의
        self.tokenizer = tokenizer  # ◀◀◀ 토크나이저 주입 필수
        self.max_len = max_len
        self.sep_id = tokenizer.piece_to_id("[SEP]")
        self.cls_id = tokenizer.piece_to_id("[CLS]")
        self.mask_id = tokenizer.piece_to_id("[MASK]")
        self.pad_id = 0

    def _apply_mlm(self, input_ids):
        """내부용 메서드: 80-10-10 마스킹 로직"""
        labels = input_ids.clone() 
        probability_matrix = torch.full(labels.shape, 0.15)
        
        # 특수 토큰 제외 마스크 생성
        # tokenizer.get_special_tokens_mask 가 없다면 수동으로 생성
        special_tokens_mask = (input_ids == self.cls_id) | \
                              (input_ids == self.sep_id) | \
                              (input_ids == self.pad_id)
        
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        masked_indices = torch.bernoulli(probability_matrix).bool()
        
        # 정답지 생성: 마스킹 안 된 곳은 -100
        labels[~masked_indices] = -100 

        # 80%는 [MASK]로 교체
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
        input_ids[indices_replaced] = self.mask_id

        # 10%는 랜덤 토큰으로 교체
        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        input_ids[indices_random] = random_words[indices_random]

        # 나머지 10%는 원본 유지 (변경 없음)
        return input_ids, labels # ◀◀◀ 변수명 일치시킴

    def __len__(self):
        return self.total_sentences - 1

    def __getitem__(self, i):
        # [NSP 로직] IsNext 결정 및 Sentence A/B 로드
        is_next = 1 if (random.random() < 0.5 and i not in self.eod_map) else 0

        raw_a = self.data[i]
        sent_a = raw_a[raw_a != self.pad_id]

        if is_next == 1:
            raw_b = self.data[i + 1]
        else:
            j = random.randint(0, self.total_sentences - 1)
            while j == i + 1 or j == i:
                j = random.randint(0, self.total_sentences - 1)
            raw_b = self.data[j]
        
        sent_b_valid = raw_b[raw_b != self.pad_id]
        sent_b = sent_b_valid[1:] 

        # [3중 트랙 조립]
        combined = np.concatenate([sent_a, sent_b])
        input_ids = combined[:self.max_len]
        padding_len = self.max_len - len(input_ids)
        
        # 1. Input IDs & Attention Mask
        input_ids = np.concatenate([input_ids, [self.pad_id] * padding_len])
        attention_mask = np.concatenate([[1] * (self.max_len - padding_len), [0] * padding_len])
        
        # 2. Token Type IDs
        token_type_ids = np.zeros(self.max_len, dtype=np.int32)
        sep_indices = np.where(input_ids == self.sep_id)[0]
        if len(sep_indices) >= 1:
            first_sep_pos = sep_indices[0]
            # 실제 문장이 끝나는 지점까지만 1로 채움
            token_type_ids[first_sep_pos + 1 : len(combined)] = 1

        # [MLM 적용 및 최종 반환]
        input_ids_tensor = torch.tensor(input_ids, dtype=torch.long)
        masked_input_ids, mlm_labels = self._apply_mlm(input_ids_tensor)

        return {
            "input_ids": masked_input_ids,
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "next_sentence_label": torch.tensor(is_next, dtype=torch.long),
            "mlm_labels": mlm_labels
        }

In [4]:
from torch.utils.data import DataLoader

# 1. 하이퍼파라미터 설정
batch_size = 48 # 한 번에 모델에게 먹일 데이터 개수
num_workers = 0 # 데이터 로딩에 사용할 CPU 코어 수 (본인 CPU 사양에 맞춰 조절)

# 2. 데이터셋 인스턴스 생성 (tokenizer는 이미 정의되어 있다고 가정)
train_dataset = BERTDataset(bin_path, eod_path, tokenizer=tokenizer_model)

# 3. 데이터로더 설정
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,       # 에폭마다 데이터 순서를 섞어 학습 효율 극대화
    num_workers=num_workers,
    pin_memory=True,     # GPU로 데이터를 더 빠르게 전송하기 위한 설정 (CUDA 사용 시)
    drop_last=True       # 마지막에 남는 짜투리 배치를 버림 (Batch Size 불일치 방지)
)

# 4. 배치 확인 루프 (제대로 나오는지 1개만 확인)
for batch in train_loader:
    print("--- 배치의 형상(Shape) 확인 ---")
    for key, val in batch.items():
        print(f"{key:20} : {val.shape}")
    
    # 여기서 첫 번째 배치를 확인했으니 바로 탈출
    break

--- 배치의 형상(Shape) 확인 ---
input_ids            : torch.Size([48, 256])
attention_mask       : torch.Size([48, 256])
token_type_ids       : torch.Size([48, 256])
next_sentence_label  : torch.Size([48])
mlm_labels           : torch.Size([48, 256])


In [5]:
import torch
import torch.nn as nn

VOCAB_SIZE = 32000
MODEL_DIMENSION = 512
MAX_LEN = 256

class Embedding(nn.Module):

    def __init__(self, vocab_size, model_dimension, token_type_ids_num, dropout: float = 0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, model_dimension)
        self.seg_embedding = nn.Embedding(token_type_ids_num, model_dimension)
        self.position_embedding = nn.Embedding(MAX_LEN, model_dimension)

        self.layer_norm = nn.LayerNorm(model_dimension)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("pos_ids", torch.arange(MAX_LEN).expand(1, -1))

    def forward(self, input_ids, token_type_ids):
        post_token_embedding = self.token_embedding(input_ids)
        post_seg_embedding = self.seg_embedding(token_type_ids)
        post_position_embedding = self.position_embedding(self.pos_ids[:, :input_ids.size(1)])

        combined_embedding = post_token_embedding + post_seg_embedding + post_position_embedding
        
        return self.dropout(self.layer_norm(combined_embedding))

batch = next(iter(train_loader))
input_ids = batch["input_ids"]
token_type_ids = batch["token_type_ids"]
embedding = Embedding(VOCAB_SIZE, MODEL_DIMENSION, 2)
out = embedding(input_ids, token_type_ids)
out


tensor([[[ 0.0000, -0.1866,  0.7169,  ...,  1.0738,  0.9716,  0.5749],
         [-0.8287, -0.0000, -0.0805,  ...,  0.7181,  0.2403, -0.0000],
         [ 0.9883,  0.6205,  0.1025,  ..., -0.0746,  1.3075,  0.0000],
         ...,
         [-0.4158, -0.3245, -0.3861,  ..., -0.2917,  1.1140,  0.9382],
         [-0.7091, -0.6380, -1.2291,  ...,  0.1006, -0.2568,  1.1252],
         [ 0.1841, -0.0000, -1.1735,  ..., -0.4554,  2.0178,  0.3311]],

        [[ 0.5538, -0.1866,  0.7169,  ...,  1.0738,  0.9716,  0.5749],
         [-0.5899, -1.8968,  2.2520,  ...,  0.8092,  1.1310, -2.8258],
         [ 0.0180, -0.3345, -0.1857,  ...,  0.7172,  0.3764,  0.0000],
         ...,
         [-0.4158, -0.0000, -0.3861,  ..., -0.2917,  0.0000,  0.9382],
         [-0.7091, -0.6380, -1.2291,  ...,  0.1006, -0.2568,  1.1252],
         [ 0.0000, -0.8067, -1.1735,  ..., -0.4554,  2.0178,  0.3311]],

        [[ 0.5538, -0.1866,  0.7169,  ...,  1.0738,  0.9716,  0.5749],
         [-0.6052, -2.1405,  2.0956,  ..., -0

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math


NUM_HEAD = 8
class Attention(nn.Module):

    def __init__(self, num_head, max_len, model_dimension, dropout: float = 0.1):
        super().__init__()
        self.max_len = max_len
        self.model_dimension = model_dimension
        self.num_head = num_head
        self.key_dimension = self.model_dimension // self.num_head

        self.query_weights = nn.Linear(self.model_dimension, self.model_dimension)
        self.key_weights = nn.Linear(self.model_dimension, self.model_dimension)
        self.value_weights = nn.Linear(self.model_dimension, self.model_dimension)
        self.out_weights = nn.Linear(self.model_dimension, self.model_dimension)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        batch_size, sequence_len, _ = x.size()


        query = self.query_weights(x).view(batch_size, sequence_len, self.num_head, self.key_dimension).transpose(1, 2)
        key = self.key_weights(x).view(batch_size, sequence_len, self.num_head, self.key_dimension).transpose(1, 2)
        value = self.value_weights(x).view(batch_size, sequence_len, self.num_head, self.key_dimension).transpose(1, 2)

        attention_scores = torch.matmul(query, key.transpose(-2, -1))
        scaled_attention_scores = attention_scores / math.sqrt(self.key_dimension)

        if mask is not None:
            # 포인트 1 & 2 통합 해결:
            # 1. mask[:, :sequence_len] -> 마스크를 현재 입력 x의 실제 길이에 맞춤 (32 vs 256 해결)
            # 2. .unsqueeze(1).unsqueeze(2) -> (Batch, Seq)를 (Batch, 1, 1, Seq)로 확장 (4D 브로드캐스팅 해결)
            mask = mask[:, :sequence_len].unsqueeze(1).unsqueeze(2)
            
            scaled_attention_scores = scaled_attention_scores.masked_fill(mask == 0, value=-1e4)

        matching_weights = F.softmax(scaled_attention_scores, dim=-1)
        matching_weights = self.dropout(matching_weights)

        out = torch.matmul(matching_weights, value)
        out = out.transpose(1, 2).contiguous().view(batch_size, sequence_len, self.model_dimension)

        out = self.out_weights(out)

        return self.dropout(out)



In [7]:
import torch.nn as nn 
import torch.nn.functional as F


class PositionWiseFeedForwardNetwork(nn.Module):

    def __init__(self, model_dimension, dropout: float = 0.1):
        super().__init__()
        self.layer1 = nn.Linear(model_dimension, model_dimension * 4)
        self.layer2 = nn.Linear(model_dimension * 4, model_dimension)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.layer2(x)

        return self.dropout(x)

In [8]:
import torch.nn as nn


class EncoderBlock(nn.Module):
    
    def __init__(self, num_head, max_len, model_dimension, dropout: float = 0.1):
        super().__init__()
        self.attention = Attention(num_head, max_len, model_dimension)
        self.position_wise_feed_forward_network = PositionWiseFeedForwardNetwork(model_dimension)
        self.norm1 = nn.LayerNorm(model_dimension)
        self.norm2 = nn.LayerNorm(model_dimension)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attention_out = self.attention(x, mask)
        x = self.norm1(self.dropout(attention_out) + x)

        ffn_out = self.position_wise_feed_forward_network(x)
        x = self.norm2(self.dropout(ffn_out) + x)

        return x

In [9]:
import torch.nn as nn

class BERT(nn.Module):

    def __init__(self, vocab_size, num_head, max_len, model_dimension, num_token_type_ids, num_layers):
        super().__init__()
        self.embedding = Embedding(vocab_size, model_dimension, num_token_type_ids)
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(num_head, max_len, model_dimension)
            for _ in range(num_layers)
        ])

    def forward(self, input_ids, token_type_ids, mask):
        x = self.embedding(input_ids, token_type_ids)
        
        for block in self.encoder_blocks:
            x = block(x, mask)

        return x

In [10]:
import torch
import torch.nn as nn

# 1. Pooler: 문장 전체의 의미를 응축
class Pooler(nn.Module):
    def __init__(self, model_dimension):
        super().__init__()
        self.linear = nn.Linear(model_dimension, model_dimension)
        self.activation = nn.Tanh() # BERT 논문의 정석

    def forward(self, x):
        # x: [Batch, Seq, Dim] -> 첫 번째 [CLS] 토큰만 추출
        first_token = x[:, 0]
        return self.activation(self.linear(first_token))

# 2. 통합 BERT 모델 (Backbone + Heads)
class BERTLM(nn.Module):
    def __init__(self, vocab_size, num_head, max_len, model_dimension, num_token_type_ids, num_layers):
        super().__init__()
        # 우리가 만든 백본
        self.bert = BERT(vocab_size, num_head, max_len, model_dimension, num_token_type_ids, num_layers)
        
        # Heads
        self.pooler = Pooler(model_dimension)
        self.mlm_head = nn.Linear(model_dimension, vocab_size) # 각 토큰별 단어 예측
        self.nsp_head = nn.Linear(model_dimension, 2)         # 문장 연속성 예측 (Yes/No)

    def forward(self, input_ids, token_type_ids, mask):
        # 1. 백본 통과 (지능 추출)
        encoded_layers = self.bert(input_ids, token_type_ids, mask)
        
        # 2. MLM 예측: 모든 토큰 위치에서 단어 확률 계산
        mlm_scores = self.mlm_head(encoded_layers)
        
        # 3. NSP 예측: [CLS] 토큰을 Pooler에 넣어 문장 관계 계산
        pooled_output = self.pooler(encoded_layers)
        nsp_scores = self.nsp_head(pooled_output)
        
        return mlm_scores, nsp_scores

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())

True


In [12]:
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

num_epochs = 3

model = BERTLM(
    vocab_size=VOCAB_SIZE, 
    num_head=NUM_HEAD, 
    max_len=MAX_LEN, 
    model_dimension=MODEL_DIMENSION, 
    num_token_type_ids=2, 
    num_layers=6
).to(device)

# 1. 손실 함수 (Loss Function)
# MLM은 패딩(0)을 학습에서 제외해야 하므로 ignore_index 설정
criterion_mlm = nn.CrossEntropyLoss(ignore_index=-100)
criterion_nsp = nn.CrossEntropyLoss()

# 2. 최적화 도구 (Optimizer)
# 논문 사양: AdamW, lr=1e-4, betas=(0.9, 0.999), weight_decay=0.01
optimizer = optim.AdamW(
    model.parameters(), 
    lr=1e-4, 
    betas=(0.9, 0.999), 
    weight_decay=0.01
)

# 3. 러닝레이트 스케줄러 (Linear Warmup + Linear Decay)
# 논문 사양: 전체 학습의 10% 동안 Warmup 수행 후 선형적으로 감소
def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(
            0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps))
        )
    return LambdaLR(optimizer, lr_lambda)

# 예시 단계 설정 (본인의 데이터셋 크기에 맞춰 조정)
total_steps = len(train_loader) * num_epochs
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

In [ ]:
# 진단 코드: 루프 안에서 에러 나기 직전에 실행해 보세요
print(batch.keys())

dict_keys(['input_ids', 'attention_mask', 'token_type_ids', 'next_sentence_label', 'mlm_labels'])


In [16]:
import torch
from tqdm.autonotebook import tqdm
# PyTorch 최신 버전 권장 방식
from torch.amp import autocast, GradScaler 

scaler = GradScaler()
model.train()

for epoch in range(num_epochs):
    total_loss = 0
    running_mlm_loss = 0 # MLM 손실 누적용
    running_nsp_loss = 0 # NSP 손실 누적용
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    model.train()
    for i, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        mask = batch['attention_mask'].to(device) 
        nsp_labels = batch['next_sentence_label'].to(device)
        mlm_labels = batch['mlm_labels'].to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            mlm_scores, nsp_scores = model(input_ids, token_type_ids, mask)
            
            # MLM & NSP 개별 손실 계산
            loss_mlm = criterion_mlm(mlm_scores.view(-1, VOCAB_SIZE), mlm_labels.view(-1))
            loss_nsp = criterion_nsp(nsp_scores, nsp_labels)
            
            # 합산 손실: $L_{total} = L_{MLM} + L_{NSP}$
            loss = loss_mlm + loss_nsp

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        # 실시간 누적 (item()을 써서 텐서에서 숫자로 추출)
        total_loss += loss.item()
        running_mlm_loss += loss_mlm.item()
        running_nsp_loss += loss_nsp.item()
        
        # tqdm 상태 표시줄에 개별 손실의 '평균치'를 노출. 현재 스텝 i+1로 나눠서 평균을 표시함.
        progress_bar.set_postfix({
            'total': f"{total_loss / (i + 1):.4f}",
            'mlm': f"{running_mlm_loss / (i + 1):.4f}",
            'nsp': f"{running_nsp_loss / (i + 1):.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.6f}"
        })

    # 에포크 종료 후 최종 요약 출력
    avg_mlm = running_mlm_loss / len(train_loader)
    avg_nsp = running_nsp_loss / len(train_loader)
    print(f"Epoch [{epoch+1}] Avg_MLM: {avg_mlm:.4f}, Avg_NSP: {avg_nsp:.4f}")

torch.save(model.state_dict(), "bert_lm.pth")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_25232\20950430.py:2: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Epoch 1/3: 100%|██████████| 65040/65040 [2:34:14<00:00,  7.03it/s, total=6.9904, mlm=6.4995, nsp=0.4909, lr=0.000074]  


Epoch [1] Avg_MLM: 6.4995, Avg_NSP: 0.4909


Epoch 2/3: 100%|██████████| 65040/65040 [2:34:04<00:00,  7.04it/s, total=4.9934, mlm=4.7330, nsp=0.2604, lr=0.000037]  


Epoch [2] Avg_MLM: 4.7330, Avg_NSP: 0.2604


Epoch 3/3: 100%|██████████| 65040/65040 [2:34:12<00:00,  7.03it/s, total=4.5101, mlm=4.3086, nsp=0.2015, lr=0.000000]  


Epoch [3] Avg_MLM: 4.3086, Avg_NSP: 0.2015


In [ ]:
# # 학습까지 생겼던 트러블
# 1. 데이터 전처리 문제, 문장을 최대한 다 살리면서 의미없는 문장과 너무 긴 문장을 필터링.
# 2. 또한 인코딩 한 정수 문장토큰의 길이가 max_len에 걸리지 않도록 각 문장의 길이를 max_len / 2 슬롯크기 안에 들어가도록 조정해주어야 하며, 이때 cls sep 등의 토큰들이 모두 포함된 길이로 계산되어야 함.
# 3. 패딩 마스크 max_len /2 길이로 생성했으므로 나중에 오류가 났음. 이 역시 어텐션 클래스 작성 단계에서 sequence_len 길이로 슬라이싱 해줬어야 함. 어차피 시퀀스 길이 자체가
#     입력문장의 size 메서드 언패킹값이므로 버려지는 지점들은 모두 패딩토큰위치들뿐임.
# 4. 이렇게 했을 떄 살린 문장의 총 수는 300만에서 400만 사이로 학습이 매우 오래걸리는 상황에 처함.
# 5. 처음엔 에포크당 4시간 중반대가 걸렸음. 혼합정밀도 적용 결과 fp32에서 fp16으로 처리가능 크기가 작아져 -1e9의 값이 오류가 나서 -1e4로 낮춤, 이렇게 해도 소프트맥스를 통과시켰을때 결과는 바뀌지 않을만큼 충분히 작은 값이라 문제는 없을 것으로 보임.
# 6. 혼합정밀도 적용 결과 에포크당 2시간 40분대로 최소 30% 이상의 시간단축을 보였지만 여전히 오래걸림.
# 7. vram이 2gb 정도 남길래, 배치사이즈를 32에서 64로 두배 늘려본 결과, 공유메모리가 점유되기 시작하며, 에포크당 시간이 17시간으로 파멸적인 결과를 보여줌. - > 48로 결정정.
# 8. 배치를 원래대로 돌리고 에포크를 5로 낮춰 결과를 일단 지켜보기로 판단. 에포크 3으로 추가

In [31]:
def predict_bert(model, tokenizer, sentence_a, sentence_b=None, device='cuda', max_len=256):
    model.eval()

    # 1. 특수 토큰 ID 정의 (본인의 토크나이저 설정에 맞게 확인 필요)
    # 보통 0: PAD, 1: UNK, 2: CLS, 3: SEP, 4: MASK 순서인 경우가 많습니다.
    # 만약 다르다면 이 수치들을 본인의 설정에 맞게 수정하세요.
    CLS_ID = tokenizer.cls_id() if hasattr(tokenizer, 'cls_id') else 2
    SEP_ID = tokenizer.sep_id() if hasattr(tokenizer, 'sep_id') else 3
    MASK_ID = tokenizer.mask_id() if hasattr(tokenizer, 'mask_id') else 4
    PAD_ID = tokenizer.pad_id() if hasattr(tokenizer, 'pad_id') else 0

    # 2. 문장 인코딩 (SentencePiece는 리스트를 반환함)
    tokens_a = tokenizer.encode_as_ids(sentence_a)
    tokens_b = tokenizer.encode_as_ids(sentence_b) if sentence_b else []

    # 3. BERT 포맷 구성: [CLS] + A + [SEP] (+ B + [SEP])
    input_ids = [CLS_ID] + tokens_a + [SEP_ID]
    token_type_ids = [0] * len(input_ids)

    if tokens_b:
        input_ids += tokens_b + [SEP_ID]
        token_type_ids += [1] * (len(tokens_b) + 1)

    # 4. 패딩 및 마스크 생성
    padding_len = max_len - len(input_ids)
    attention_mask = [1] * len(input_ids) + [0] * padding_len
    input_ids += [PAD_ID] * padding_len
    token_type_ids += [0] * padding_len

    # 5. 텐서 변환
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    token_type_ids = torch.tensor([token_type_ids], dtype=torch.long).to(device)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long).to(device)

    # 6. 추론
    with torch.no_grad():
        mlm_scores, nsp_scores = model(input_ids, token_type_ids, attention_mask)

    # --- 이후 MLM/NSP 해석 로직은 이전과 동일 ---
    mask_token_index = (input_ids == MASK_ID).nonzero(as_tuple=True)[1]
    
    predicted_sentences = []
    if len(mask_token_index) > 0:
        for idx in mask_token_index:
            logits = mlm_scores[0, idx]
            probs = F.softmax(logits, dim=-1)
            top_k = torch.topk(probs, k=5)
            top_words = [tokenizer.id_to_piece(int(i)) for i in top_k.indices] # decode 대신 id_to_piece 사용
            predicted_sentences.append(top_words)

    nsp_probs = F.softmax(nsp_scores, dim=-1)
    is_next = torch.argmax(nsp_probs, dim=-1).item() == 0
    next_prob = nsp_probs[0, 0].item()

    return predicted_sentences, is_next, next_prob

# model = BERTLM(...) # 기존 모델 선언
model.load_state_dict(torch.load("bert_lm.pth"))
model.to(device)


# 문장 A와 문장 B를 콤마(,)로 분리해서 전달하세요.
sentence_a = "다음 식은 [MASK]의 주기성을 나타낸다."
sentence_b = "다음 식은 삼각함수의 대칭성을 나타낸다."

mlm_res, nsp_res, nsp_prob = predict_bert(model, tokenizer_model, sentence_a, sentence_b)

print(f"문장 A: {sentence_a}")
print(f"문장 B: {sentence_b}")
print(f"MLM 추천 (빈칸): {mlm_res}")
print(f"NSP 결과: {'연속된 문장(IsNext)' if nsp_res else '관계 없는 문장(NotNext)'}")
print(f"연속 확률: {nsp_prob:.4f}")

문장 A: 다음 식은 [MASK]의 주기성을 나타낸다.
문장 B: 다음 식은 삼각함수의 대칭성을 나타낸다.
MLM 추천 (빈칸): [['<unk>', 'q', '펫', '퀼', '샷']]
NSP 결과: 관계 없는 문장(NotNext)
연속 확률: 0.0038


In [32]:
# 1. 본인의 토크나이저에서 진짜 ID를 직접 찍어보세요 (가장 중요)
print(f"CLS ID: {tokenizer_model.piece_to_id('[CLS]')}")
print(f"SEP ID: {tokenizer_model.piece_to_id('[SEP]')}")
print(f"MASK ID: {tokenizer_model.piece_to_id('[MASK]')}")

# 2. 강제 정렬 추론 함수
def predict_bert_fixed(model, tokenizer, sentence_a, sentence_b, device='cuda'):
    model.eval()
    
    # [MASK]를 제외한 순수 텍스트만 인코딩
    text_a_clean = sentence_a.replace("[MASK]", " ").split(" ")
    # 실제로는 [MASK] 자리를 비우고 인코딩한 뒤 수동으로 MASK_ID를 박아야 합니다.
    
    # 더 확실한 방법: 수동 토큰 구성
    # (학습 때 썼던 특이한 구조 [CLS] A [SEP] [CLS] B [SEP]를 그대로 복원)
    tokens_a = tokenizer.encode_as_ids(sentence_a.replace("[MASK]", "")) 
    # 주의: 위 방식보다 [MASK] 위치를 정확히 찾아서 ID를 직접 삽입해야 함
    
    # 종하님, 지금 당장 아래 코드로 'ID'가 학습 때와 맞는지부터 확인합시다.

CLS ID: 2
SEP ID: 3
MASK ID: 4


In [46]:
def predict_bert_aligned(model, tokenizer, sentence_a, sentence_b, device='cuda', max_len=256):
    model.eval()
    CLS_ID, SEP_ID, MASK_ID, PAD_ID = 2, 3, 4, 0

    # 1. 문장 A 구성 ([CLS] + text + [SEP])
    part1, part2 = sentence_a.split("[MASK]")
    ids_a = [CLS_ID] + tokenizer.encode_as_ids(part1) + [MASK_ID] + tokenizer.encode_as_ids(part2) + [SEP_ID]
    
    # 2. 문장 B 구성 (훈련 데이터셋 로직에 따라 [CLS] 없이 text + [SEP])
    # sent_b = sent_b_valid[1:] 로직을 반영하여 CLS를 붙이지 않습니다.
    ids_b = tokenizer.encode_as_ids(sentence_b) + [SEP_ID]
    
    # 3. 전체 합치기
    input_ids = ids_a + ids_b
    token_type_ids = [0] * len(ids_a) + [1] * len(ids_b)
    
    # 4. 패딩 및 텐서화
    padding_len = max_len - len(input_ids)
    attention_mask = [1] * len(input_ids) + [0] * padding_len
    input_ids += [PAD_ID] * padding_len
    token_type_ids += [0] * padding_len
    
    input_ids = torch.tensor([input_ids]).to(device)
    token_type_ids = torch.tensor([token_type_ids]).to(device)
    attention_mask = torch.tensor([attention_mask]).to(device)
    
    with torch.no_grad():
        mlm_scores, nsp_scores = model(input_ids, token_type_ids, attention_mask)
    
    # MLM 해석
    mask_idx = (input_ids == MASK_ID).nonzero(as_tuple=True)[1]
    mlm_res = [[tokenizer.id_to_piece(int(i)) for i in torch.topk(torch.softmax(mlm_scores[0, idx], dim=-1), k=5).indices] for idx in mask_idx]
        
    # NSP 해석 (종하님의 데이터셋은 1이 IsNext입니다!)
    nsp_probs = torch.softmax(nsp_scores, dim=-1)
    is_next = torch.argmax(nsp_probs, dim=-1).item() == 1 # ◀◀◀ 1번 라벨 확인
    
    return mlm_res, is_next, nsp_probs[0, 1].item() # ◀◀◀ 1번(IsNext) 확률 반환


sentence_a = "너의 [MASK] 뭐니?."
sentence_b = "너의 나이는 몇살이니?."
    

# 실행
# 함수 호출 (반환 변수명을 출력 변수명과 일치시킴)
# 함수 이름 일치시켜서 호출!
mlm_res, nsp_res, nsp_prob = predict_bert_aligned(model, tokenizer_model, sentence_a, sentence_b)

print(f"문장 A: {sentence_a}")
print(f"문장 B: {sentence_b}")
print(f"MLM 추천 (빈칸): {mlm_res}")
print(f"NSP 결과: {'연속된 문장(IsNext)' if nsp_res else '관계 없는 문장(NotNext)'}")
print(f"연속 확률: {nsp_prob:.4f}")

문장 A: 너의 [MASK] 뭐니?.
문장 B: 너의 나이는 몇살이니?.
MLM 추천 (빈칸): [['▁나이는', '▁아내는', '▁아버지는', '▁뜻을', '▁말을']]
NSP 결과: 연속된 문장(IsNext)
연속 확률: 0.9996
